# Docs ETL: raw Markdown → sections dataset

Runs the `rag_etl` pipeline step by step over `docs_raw/`:

1. **Load** every `.md` file (`<project_name>/<version>/**/*.md`).
2. **Drop navigation files** (sidebars, coverpages), detected from their structure, not their file names.
3. **Split into sections** on Markdown (`#`) and HTML (`<h1>`..`<h6>`) headings.
4. **Flag `is_TO_DO`**: sections whose body is only a placeholder.
5. **Flag `For_training` with MinHash LSH**: find cross-version near-duplicates; only the latest version of each group is kept for training.
6. **Semantic dedup with EmbeddingGemma** and a comparison with MinHash LSH (results and run time).
7. **Save** the dataset, flagged by the method chosen in `METHOD`.

> Step 6 downloads `google/embeddinggemma-300m`, which is gated on Hugging Face: accept its license on the model page and authenticate (`hf auth login` or `export HF_TOKEN=...`) before running it.

In [1]:
import time
from pathlib import Path

import pandas as pd

from rag_etl.compare import compare_flags, compare_pairs, time_minhash, time_semantic
from rag_etl.flags import DEFAULT_THRESHOLDS, find_minhash_duplicate_pairs, flag_near_duplicates, flag_todo
from rag_etl.navigation import is_navigation_file, navigation_stats
from rag_etl.parse_docs import (
    COLUMNS,
    FLAG_COLUMNS,
    drop_navigation_files,
    explode_sections,
    load_files,
    save_dataset,
)
from rag_etl.semantic import DEFAULT_MODEL, find_semantic_duplicate_pairs, load_model

pd.set_option("display.max_colwidth", 80)

DOCS_DIR = Path("docs_raw")                  # relative to the repo root
OUTPUT_PATH = Path("data/sections.parquet")
METHOD = "minhash"                           # which dedup sets the saved flags: "minhash" | "semantic"
MINHASH_THRESHOLD = DEFAULT_THRESHOLDS["minhash"]    # estimated Jaccard (char 5-shingles)
SEMANTIC_THRESHOLD = DEFAULT_THRESHOLDS["semantic"]  # cosine similarity
MODEL_NAME = DEFAULT_MODEL

## 1. Load files

In [2]:
files = load_files(DOCS_DIR)
print(f"{len(files)} files")
files.groupby(["project_name", "version"]).size().rename("n_files").to_frame()

88 files


n_files
project_name                version                           
2p-revenue-optimizer-api    0.2.2                            3
                            latest                           4
catalog-portfolio-api       0.0.1-test-bench-v2a             9
                            latest                           9
demand-forecast-docs        latest                           5
order-workflow-api          5.6.2-hotfix-1                   9
payment-promise-gateway     0.0.10-test-crc-migration        1
                            latest                           1
price-engine-api            latest                           6
traffic-gate-api            latest                           7
unified-transaction-configs v0.0.2                           3
vendor-stockkeeper-api      0.0.10-stock-consumer            7
                            0.0.3-paused-seller              7
                            0.0.6-ep-restock-man             7
                            latest                           7
zenith-keeper               1.1.6                            3

## 2. Drop navigation files

A file counts as navigation when it has at most 1 heading, at least 50% of its lines are links/images, and it has at most 30 words of plain text.

In [3]:
stats = pd.DataFrame([navigation_stats(md) for md in files["markdown"]], index=files["path"])
stats["is_navigation"] = files["markdown"].map(is_navigation_file).values
stats[stats["is_navigation"]]

,n_lines,n_headings,link_ratio,prose_words,is_navigation
path,,,,,
catalog-portfolio-api/0.0.1-test-bench-v2a/guide/_sidebar.md,8,0,1.000000,0,True
catalog-portfolio-api/latest/guide/_sidebar.md,8,0,1.000000,0,True
demand-forecast-docs/latest/guide/_sidebar.md,5,0,0.800000,1,True
order-workflow-api/5.6.2-hotfix-1/guide/_sidebar.md,13,0,0.846154,2,True
price-engine-api/latest/guide/_coverpage.md,6,1,0.666667,14,True
price-engine-api/latest/guide/_sidebar.md,6,0,0.833333,3,True
traffic-gate-api/latest/guide/_sidebar.md,6,0,0.833333,3,True
traffic-gate-api/latest/guide/coverpage.md,6,1,0.500000,21,True
unified-transaction-configs/v0.0.2/guide/_sidebar.md,5,0,1.000000,0,True


In [4]:
content_files = drop_navigation_files(files)
print(f"Dropped {len(files) - len(content_files)} navigation files, kept {len(content_files)}")

Dropped 18 navigation files, kept 70


## 3. Split into sections

In [5]:
sections = explode_sections(content_files)
print(f"{len(sections)} sections")
sections.head()

461 sections


,path,file_name,project_name,version,section,text
0,2p-revenue-optimizer-api/0.2.2/guide/README.md,README.md,2p-revenue-optimizer-api,0.2.2,Spring Boot App model for Java 17,We provide a basic model for JDK 17 / Spring based web applications.\n\nPlea...
1,2p-revenue-optimizer-api/0.2.2/guide/README.md,README.md,2p-revenue-optimizer-api,0.2.2,Spring Boot App model for Java 17 > Usage > SCOPE,The suffix of each VendorFlow **SCOPE** is used to know which properties fil...
2,2p-revenue-optimizer-api/0.2.2/guide/README.md,README.md,2p-revenue-optimizer-api,0.2.2,Spring Boot App model for Java 17 > Usage > Web Server,Each Spring Boot web application includes an embedded web server. For servle...
3,2p-revenue-optimizer-api/0.2.2/guide/README.md,README.md,2p-revenue-optimizer-api,0.2.2,Spring Boot App model for Java 17 > Usage > Main,"The main class for this app is Application, where Spring context is initiali..."
4,2p-revenue-optimizer-api/0.2.2/guide/README.md,README.md,2p-revenue-optimizer-api,0.2.2,Spring Boot App model for Java 17 > Usage > Error Handling,We also provide basic handling for exceptions in ControllerExceptionHandler ...


In [6]:
# HTML headings now split sections too, e.g. the order-workflow README
sections[sections["path"] == "order-workflow-api/5.6.2-hotfix-1/guide/README.md"][["section", "text"]]

,section,text
264,,"<p>\n <a href=""https://github.com/retailcorp/flow_order-configs-api"" rel=..."
265,Release Process - Workflow API,"<p>\n <a href=""https://app.datadoghq.com/dashboard/rs7-pqm-yzt/retailcorp..."
266,Getting started,Workflow API simplifies and standardizes the development flow of application...
267,Components > Database,A MySQL database is used to store the following information:\n\n* Workflow c...
268,Components > BigQ,A BigQ Consumer is used to receive tasks from the [Orchestrator](http://reta...
269,Components > Jobs,The Jobs service is used to automatically create versions based on a cron-li...


## 4. `is_TO_DO` flag

In [7]:
sections = flag_todo(sections)
sections[sections["is_TO_DO"] == 1][["path", "section", "text"]]

,path,section,text
283,order-workflow-api/5.6.2-hotfix-1/guide/services/database.md,Database > Objects stored > Cron History,_TODO_
353,price-engine-api/latest/guide/general.md,General > Introducción,TO DO.
354,price-engine-api/latest/guide/general.md,General > Responsabilidades > Price Engine API es responsable de:,TO DO.
355,price-engine-api/latest/guide/general.md,General > Responsabilidades > Price Engine API no se responsabiliza por:,TO DO.


## 5. Near-duplicates with MinHash LSH → `For_training`

Each section's character 5-shingles are hashed into a 128-permutation MinHash signature. LSH buckets the signatures so only likely pairs (around the threshold) are proposed, instead of comparing every pair.
Each proposed pair is scored with the MinHash estimate of Jaccard similarity. Within each project, every section keeps only its best match in each other version.
Pairs at or above `MINHASH_THRESHOLD` are linked into groups (`near_dup_group`). Within a group, only rows from the latest version get `For_training = 1` (`latest` beats any number; otherwise versions are compared numerically).
Sections with no near-duplicate keep `For_training = 1`.

In [8]:
# Best-match similarities with a low LSH threshold: genuine duplicates score
# >= 0.8, the closest non-duplicate ~0.6, so the threshold sits in that gap.
low_pairs = find_minhash_duplicate_pairs(sections, threshold=0.3)
pd.cut(low_pairs["similarity"], [0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0], include_lowest=True).value_counts().sort_index()

similarity
(0.299, 0.5]      0
(0.5, 0.6]        0
(0.6, 0.7]        1
(0.7, 0.8]        0
(0.8, 0.9]        3
(0.9, 0.95]       7
(0.95, 1.0]     262
Name: count, dtype: int64

In [9]:
sections_minhash = flag_near_duplicates(sections, method="minhash", threshold=MINHASH_THRESHOLD)
(
    sections_minhash.groupby(["project_name", "version"])
    .agg(
        sections=("text", "size"),
        in_dup_group=("near_dup_group", "count"),
        for_training=("For_training", "sum"),
    )
)

sections  in_dup_group  \
project_name                version                                             
2p-revenue-optimizer-api    0.2.2                            12            12   
                            latest                           13            12   
catalog-portfolio-api       0.0.1-test-bench-v2a            107           107   
                            latest                          107           107   
demand-forecast-docs        latest                           25             0   
order-workflow-api          5.6.2-hotfix-1                   28             0   
payment-promise-gateway     0.0.10-test-crc-migration        27            27   
                            latest                           27            27   
price-engine-api            latest                           10             0   
traffic-gate-api            latest                           13             0   
unified-transaction-configs v0.0.2                            5             0   
vendor-stockkeeper-api      0.0.10-stock-consumer            22            21   
                            0.0.3-paused-seller              21            21   
                            0.0.6-ep-restock-man             21            21   
                            latest                           21            21   
zenith-keeper               1.1.6                             2             0   

                                                       for_training  
project_name                version                                  
2p-revenue-optimizer-api    0.2.2                                 0  
                            latest                               13  
catalog-portfolio-api       0.0.1-test-bench-v2a                  0  
                            latest                              107  
demand-forecast-docs        latest                               25  
order-workflow-api          5.6.2-hotfix-1                       28  
payment-promise-gateway     0.0.10-test-crc-migration             0  
                            latest                               27  
price-engine-api            latest                               10  
traffic-gate-api            latest                               13  
unified-transaction-configs v0.0.2                                5  
vendor-stockkeeper-api      0.0.10-stock-consumer                 1  
                            0.0.3-paused-seller                   0  
                            0.0.6-ep-restock-man                  0  
                            latest                               21  
zenith-keeper               1.1.6                                 2

In [10]:
# Example group: same section across the 4 vendor-stockkeeper-api versions
example = sections_minhash[sections_minhash["section"].str.contains("Validation stock", na=False)]["near_dup_group"].iloc[0]
sections_minhash[sections_minhash["near_dup_group"] == example][["version", "section", "For_training"]]

,version,section,For_training
378,0.0.10-stock-consumer,Alerts > How to resolve > [LOW PRIORITY] Validation stock (publication has m...,0
400,0.0.3-paused-seller,Alerts > How to resolve > [LOW PRIORITY] Validation stock (publication has m...,0
421,0.0.6-ep-restock-man,Alerts > How to resolve > Validation stock (publication has more stock than ...,0
442,latest,Alerts > How to resolve > [LOW PRIORITY] Validation stock (publication has m...,1


## 6. Semantic near-duplicates with EmbeddingGemma vs MinHash LSH

Sections are embedded with `google/embeddinggemma-300m`, using its `STS` prompt for text-to-text similarity, and L2-normalized, so cosine similarity is a dot product.
Pairs go through the same best-match selection and grouping as MinHash; only the similarity measure changes.
Cosine and Jaccard are on different scales, so the semantic threshold is calibrated separately below.

In [11]:
start = time.perf_counter()
model = load_model(MODEL_NAME)
model_load_s = time.perf_counter() - start
print(f"Loaded {MODEL_NAME} on {model.device} in {model_load_s:.1f}s")

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Loaded google/embeddinggemma-300m on cuda:0 in 5.7s


### Run time
Median of 3 runs. Embedding is warmed up once first, so CUDA start-up isn't counted. Model loading is a one-time cost, shown separately.

In [12]:
mh = time_minhash(sections, repeats=3, threshold=MINHASH_THRESHOLD)
sem = time_semantic(sections, model, repeats=3, threshold=SEMANTIC_THRESHOLD)
embeddings = sem["embeddings"]

pd.DataFrame(
    {
        "MinHash LSH": {"model load (once)": 0.0, "signatures / encode": None, "search": None, "total (excl. load)": mh["seconds_total"]},
        f"Semantic ({model.device})": {
            "model load (once)": model_load_s,
            "signatures / encode": sem["seconds_encode"],
            "search": sem["seconds_search"],
            "total (excl. load)": sem["seconds_total"],
        },
    }
).round(3)

,MinHash LSH,Semantic (cuda:0)
model load (once),0.000,5.708
signatures / encode,NaN,3.889
search,NaN,0.009
total (excl. load),0.095,3.898


### Calibrating the semantic threshold
Best-match cosine score of every section against each other version of its project (no threshold). The threshold should sit in the gap between true duplicates and the closest non-duplicates.

In [13]:
best_cos = find_semantic_duplicate_pairs(sections, threshold=-1.0, embeddings=embeddings)
print(pd.cut(best_cos["similarity"], [-1, 0.5, 0.7, 0.8, 0.85, 0.9, 0.95, 0.98, 1.0001]).value_counts().sort_index())

# Lowest-scoring best matches: where the gap is
lowest = best_cos.nsmallest(15, "similarity").assign(
    left_version=lambda d: sections.loc[d["left"], "version"].values,
    left_section=lambda d: sections.loc[d["left"], "section"].str[-50:].values,
    right_version=lambda d: sections.loc[d["right"], "version"].values,
    right_section=lambda d: sections.loc[d["right"], "section"].str[-50:].values,
)
lowest

similarity
(-1.0, 0.5]       0
(0.5, 0.7]        0
(0.7, 0.8]        0
(0.8, 0.85]       0
(0.85, 0.9]       3
(0.9, 0.95]       1
(0.95, 0.98]      3
(0.98, 1.0]     269
Name: count, dtype: int64


,left,right,similarity,left_version,left_section,right_version,right_section
177,384,404,0.868348,0.0.10-stock-consumer,Domain > Services > Operation Validation Consumer,0.0.3-paused-seller,Domain > Services > Restock Consumer
178,384,425,0.868348,0.0.10-stock-consumer,Domain > Services > Operation Validation Consumer,0.0.6-ep-restock-man,Domain > Services > Restock Consumer
179,384,446,0.868348,0.0.10-stock-consumer,Domain > Services > Operation Validation Consumer,latest,Domain > Services > Restock Consumer
11,11,22,0.936563,0.2.2,rule_row Table,latest,rulecustomfield Table
160,378,421,0.968196,0.0.10-stock-consumer,(publication has more stock than warehouse's stock,0.0.6-ep-restock-man,(publication has more stock than warehouse's stock
221,400,421,0.968196,0.0.3-paused-seller,(publication has more stock than warehouse's stock,0.0.6-ep-restock-man,(publication has more stock than warehouse's stock
259,421,442,0.968196,0.0.6-ep-restock-man,(publication has more stock than warehouse's stock,latest,(publication has more stock than warehouse's stock
168,381,403,0.995142,0.0.10-stock-consumer,Domain > Services,0.0.3-paused-seller,Domain > Services
169,381,424,0.995142,0.0.10-stock-consumer,Domain > Services,0.0.6-ep-restock-man,Domain > Services
170,381,445,0.995142,0.0.10-stock-consumer,Domain > Services,latest,Domain > Services


In [14]:
sections_semantic = flag_near_duplicates(
    sections, method="semantic", threshold=SEMANTIC_THRESHOLD, embeddings=embeddings
)
(
    sections_semantic.groupby(["project_name", "version"])
    .agg(
        sections=("text", "size"),
        in_dup_group=("near_dup_group", "count"),
        for_training=("For_training", "sum"),
    )
)

sections  in_dup_group  \
project_name                version                                             
2p-revenue-optimizer-api    0.2.2                            12            12   
                            latest                           13            12   
catalog-portfolio-api       0.0.1-test-bench-v2a            107           107   
                            latest                          107           107   
demand-forecast-docs        latest                           25             0   
order-workflow-api          5.6.2-hotfix-1                   28             0   
payment-promise-gateway     0.0.10-test-crc-migration        27            27   
                            latest                           27            27   
price-engine-api            latest                           10             0   
traffic-gate-api            latest                           13             0   
unified-transaction-configs v0.0.2                            5             0   
vendor-stockkeeper-api      0.0.10-stock-consumer            22            21   
                            0.0.3-paused-seller              21            21   
                            0.0.6-ep-restock-man             21            21   
                            latest                           21            21   
zenith-keeper               1.1.6                             2             0   

                                                       for_training  
project_name                version                                  
2p-revenue-optimizer-api    0.2.2                                 0  
                            latest                               13  
catalog-portfolio-api       0.0.1-test-bench-v2a                  0  
                            latest                              107  
demand-forecast-docs        latest                               25  
order-workflow-api          5.6.2-hotfix-1                       28  
payment-promise-gateway     0.0.10-test-crc-migration             0  
                            latest                               27  
price-engine-api            latest                               10  
traffic-gate-api            latest                               13  
unified-transaction-configs v0.0.2                                5  
vendor-stockkeeper-api      0.0.10-stock-consumer                 1  
                            0.0.3-paused-seller                   0  
                            0.0.6-ep-restock-man                  0  
                            latest                               21  
zenith-keeper               1.1.6                                 2

### Do the methods agree?
Pairs found by each method (each with both scores), then the effect on `For_training`.

In [15]:
pairs_cmp = compare_pairs(sections, mh["pairs"], sem["pairs"], mh["minhashes"], embeddings)
pairs_cmp["found_by"].value_counts()

found_by
both             272
only_minhash       0
only_semantic      0
Name: count, dtype: int64

In [16]:
disagreements = pairs_cmp[pairs_cmp["found_by"] != "both"]
disagreements[["found_by", "project_name", "jaccard_est", "cosine", "left_version", "left_section", "right_version", "right_section"]].round(3)

,found_by,project_name,jaccard_est,cosine,left_version,left_section,right_version,right_section


In [17]:
flags_table, agreement = compare_flags(sections_minhash, sections_semantic)
print(f"For_training agreement: {agreement:.1%}")
flags_table

For_training agreement: 100.0%


semantic,0,1
minhash,,
0,209,0
1,0,252


### How sensitive is each threshold?
Re-flag with a range of thresholds and count rows whose `For_training` differs from the MinHash result. A row moving 1 → 0 means unique content would be dropped from training.

In [18]:
def flag_diff(flagged):
    diff = flagged["For_training"] - sections_minhash["For_training"]
    return {"wrongly_dropped (1→0)": int((diff < 0).sum()), "extra_kept (0→1)": int((diff > 0).sum())}

rows = {}
for t in [0.85, 0.90, 0.93, 0.95, 0.97, 0.99]:
    rows[("semantic", t)] = flag_diff(flag_near_duplicates(sections, method="semantic", threshold=t, embeddings=embeddings))
for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
    rows[("minhash", t)] = flag_diff(flag_near_duplicates(sections, method="minhash", threshold=t))
pd.DataFrame(rows).T.rename_axis(["method", "threshold"])

wrongly_dropped (1→0)  extra_kept (0→1)
method   threshold                                         
semantic 0.85                           1                 0
         0.90                           0                 0
         0.93                           0                 0
         0.95                           0                 0
         0.97                           0                 1
         0.99                           0                 1
minhash  0.50                           0                 0
         0.60                           0                 0
         0.70                           0                 0
         0.80                           0                 1
         0.90                           0                 2

### Takeaways (docs_raw, RTX 3080 Ti)

- **Same answer at the calibrated thresholds.** Both methods find the same 272 cross-version pairs and agree on `For_training` for all 461 rows. On this data, version near-duplicates are close copies of the text, so word-level overlap is enough to catch them.
- **MinHash is ~40× faster.** About 0.1 s on CPU, versus about 4 s of GPU encoding plus a one-time 5 s model load for EmbeddingGemma. At 10M documents the embedding cost dominates, and the semantic search would also need an approximate-nearest-neighbour index instead of the per-project matrix used here. MinHash LSH already avoids comparing every pair.
- **Embeddings squeeze the score range.** EmbeddingGemma rates every section as very similar to its closest section in another version (lowest best-match cosine 0.87), even when the content differs (different consumers, different tables). Safe semantic thresholds lie only in about 0.87–0.968: at 0.85 a section that exists in only one version is wrongly dropped, and at 0.97 an edited old copy is kept. For MinHash, any threshold from at least 0.5 up to 0.79 gives the same flags.
- **Decision:** keep **MinHash LSH** as the default for deduplicating versions (`METHOD = "minhash"`). Embeddings are better spent on RAG retrieval, and on finding paraphrased duplicates across projects, which MinHash can't see.

## 7. Save
`METHOD` picks which detector's `near_dup_group` / `For_training` go into the dataset.

In [19]:
flagged = {"minhash": sections_minhash, "semantic": sections_semantic}[METHOD]
dataset = flagged[COLUMNS + FLAG_COLUMNS]
out = save_dataset(dataset, OUTPUT_PATH)
print(f"Saved {len(dataset)} rows ({METHOD}) to {out}")
print(dataset[["For_training", "is_TO_DO"]].value_counts().sort_index())
dataset.dtypes

Saved 461 rows (minhash) to data/sections.parquet
For_training  is_TO_DO
0             0           209
1             0           248
              1             4
Name: count, dtype: int64


path              string
file_name         string
project_name      string
version           string
section           string
text              string
near_dup_group     Int64
For_training        int8
is_TO_DO            int8
dtype: object